# Семинар. Численное интегрирование

**Курс «Численные методы», 2 курс физического факультета СПбГУ**

**План семинара**

Разминка: решение уравнений
0. Немного Python
1. Постановка задачи
2. Формулы прямоугольников
3. Формула трапеций
4. Формула Симпсона
5. Формула Рунге–Ромберга
6. Методика Ричардсона
7. Интегрирование в `scipy`
8. Задачи
9. Домашнее задание

**Напоминание по работе с блокнотом**

- Ячейка выполняется сочетанием **Shift + Enter**. Ячейки выполняются в том порядке, в котором вы их запускаете.
- Ошибка `NameError: name '...' is not defined` обычно означает, что пропущена одна из ячеек выше.
- Если ячейка выполняется слишком долго, остановите её: кнопка «стоп» слева от ячейки или меню **Runtime → Interrupt execution** в Colab (**Kernel → Interrupt** в Jupyter).
- Ячейки **«Пишем вместе»** заполняем вместе на семинаре. Ячейки с готовым кодом нужно просто запустить.

In [ ]:
import importlib.util

for name in ['numpy', 'scipy', 'matplotlib']:
    if importlib.util.find_spec(name) is None:
        print(f'{name}: НЕ установлен, выполните  %pip install {name}')
    else:
        print(f'{name}: OK')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True

## Разминка: решение уравнений

1. С помощью `brentq` найдите корень уравнения $x^3 + x - 1 = 0$ на отрезке $[0, 1]$.
2. Вопрос: как найти $x$, при котором
$$\int_0^x e^{-t^2}\,dt = 0.5\,?$$
Чего не хватает, чтобы применить `brentq`?

In [ ]:
# Ваш код


## 0. Немного Python

Сегодня понадобятся срезы с шагом `y[начало:конец:шаг]` и функция `np.sum`.

In [ ]:
# Пишем вместе:
# 1) массив y_demo из чисел 0, 1, ..., 8 (np.arange)
# 2) напечатать: весь массив; y_demo[1:-1]; y_demo[1:-1:2]; y_demo[2:-1:2]; сумму элементов


## 1. Постановка задачи

Требуется вычислить

$$I = \int_a^b f(x)\,dx.$$

Отрезок разбивается на $N$ равных частей: $h = \dfrac{b - a}{N}$, узлы $x_i = a + ih$, $i = 0, \dots, N$. Приближённое значение интеграла на такой сетке обозначим $U_N$.

**Порядок точности.** Формула имеет порядок $p$, если

$$I - U_N \approx C h^p.$$

При увеличении $N$ в 2 раза погрешность уменьшается в $2^p$ раз: в 2 раза при $p = 1$, в 4 раза при $p = 2$, в 16 раз при $p = 4$.

**Тестовый интеграл** на этом семинаре:

$$\int_0^{\pi/2} \sin x\,dx = 1.$$

Три способа приблизить площадь под графиком (просто запустите):

In [ ]:
N_ill = 4
a_ill, b_ill = 0.0, np.pi / 2
h_ill = (b_ill - a_ill) / N_ill
x_fine = np.linspace(a_ill, b_ill, 200)
x_nodes = np.linspace(a_ill, b_ill, N_ill + 1)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
titles = ['Левые прямоугольники', 'Средние прямоугольники', 'Трапеции']
for k in range(3):
    ax[k].plot(x_fine, np.sin(x_fine), 'k')
    ax[k].set_title(titles[k])
for i in range(N_ill):
    x_l, x_r = x_nodes[i], x_nodes[i + 1]
    x_m = 0.5 * (x_l + x_r)
    ax[0].fill([x_l, x_l, x_r, x_r], [0, np.sin(x_l), np.sin(x_l), 0], alpha=0.3, color='C0', edgecolor='C0')
    ax[0].plot(x_l, np.sin(x_l), 'o', color='C0')
    ax[1].fill([x_l, x_l, x_r, x_r], [0, np.sin(x_m), np.sin(x_m), 0], alpha=0.3, color='C1', edgecolor='C1')
    ax[1].plot(x_m, np.sin(x_m), 'o', color='C1')
    ax[2].fill([x_l, x_l, x_r, x_r], [0, np.sin(x_l), np.sin(x_r), 0], alpha=0.3, color='C2', edgecolor='C2')
ax[2].plot(x_nodes, np.sin(x_nodes), 'o', color='C2')
plt.show()

Вспомогательная функция: печатает погрешность метода для $N = 4, 8, \dots, 128$ и отношение соседних погрешностей (просто запустите).

In [ ]:
def order_table(method, f, a, b, I_exact):
    print('    N    погрешность   отношение')
    err_old = None
    for N in [4, 8, 16, 32, 64, 128]:
        err = abs(I_exact - method(f, a, b, N))
        if err_old is None:
            print(f'{N:5d}   {err:.3e}')
        else:
            print(f'{N:5d}   {err:.3e}    {err_old / err:7.2f}')
        err_old = err

## 2. Формулы прямоугольников

**Левые прямоугольники:** на каждом отрезке функция заменяется её значением в левом конце.

$$U_N = h\sum_{i=0}^{N-1} f(x_i), \qquad p = 1.$$

In [ ]:
# Пишем вместе:
# функция rect_left(f, a, b, N):
#   шаг h; узлы x_0, ..., x_{N-1} = a + h * (0, 1, ..., N-1); вернуть h * сумма f(x)


In [ ]:
# Пишем вместе:
# 1) значения для N = 10 и N = 20
# 2) их погрешности (точный ответ равен 1)
# 3) отношение погрешностей


In [ ]:
# Пишем вместе:
# таблица погрешностей: order_table(метод, функция, a, b, точное значение)


**Средние прямоугольники:** значение берётся в середине каждого отрезка.

$$U_N = h\sum_{i=0}^{N-1} f\!\left(x_i + \frac{h}{2}\right), \qquad p = 2.$$

In [ ]:
# Пишем вместе:
# 1) функция rect_mid(f, a, b, N): как rect_left, но узлы сдвинуты на h/2
# 2) таблица погрешностей


**Мини-задача.** Вычислите $\displaystyle\int_1^2 \frac{dx}{x} = \ln 2$ обоими методами при $N = 10$ и найдите погрешности (точное значение `np.log(2)`).

In [ ]:
# Ваш код


## 3. Формула трапеций

На каждом отрезке функция заменяется прямой, проходящей через значения на концах.

$$U_N = h\left(\frac{f_0}{2} + f_1 + f_2 + \dots + f_{N-1} + \frac{f_N}{2}\right), \qquad p = 2.$$

In [ ]:
# Пишем вместе:
# 1) функция trapezoid(f, a, b, N):
#      узлы x (N + 1 точка, np.linspace), значения y = f(x), шаг h;
#      вернуть h * (y[0] / 2 + сумма внутренних значений + y[-1] / 2)
# 2) таблица погрешностей


Сравним погрешности средних прямоугольников и трапеций при одном и том же $N$.

In [ ]:
# Пишем вместе:
# погрешности (со знаком) rect_mid и trapezoid при N = 10


**Мини-задача.** Вычислите $\displaystyle\int_1^2 \frac{dx}{x}$ по формуле трапеций при $N = 10$ и сравните погрешность с погрешностью средних прямоугольников.

In [ ]:
# Ваш код


## 4. Формула Симпсона (парабол)

На каждой паре соседних отрезков функция заменяется параболой, проходящей через три точки. Число отрезков $N$ должно быть **чётным**.

$$U_N = \frac{h}{3}\left(f_0 + 4f_1 + 2f_2 + 4f_3 + \dots + 2f_{N-2} + 4f_{N-1} + f_N\right), \qquad p = 4.$$

In [ ]:
# Пишем вместе:
# 1) функция simpson(f, a, b, N): начало как в trapezoid;
#      вернуть h / 3 * (y[0] + 4 * сумма y с нечётными номерами + 2 * сумма внутренних y с чётными номерами + y[-1])
# 2) таблица погрешностей


Проверим догадку из раздела 3: комбинация средних прямоугольников и трапеций на сетке из $N$ отрезков совпадает с формулой Симпсона на сетке из $2N$ отрезков.

In [ ]:
# Пишем вместе:
# (2 * rect_mid + trapezoid) / 3 при N = 10 минус simpson при N = 20


**Мини-задача.** Вычислите $\displaystyle\int_1^2 \frac{dx}{x}$ по формуле Симпсона при $N = 10$. Во сколько раз погрешность меньше, чем у формулы трапеций?

In [ ]:
# Ваш код


Погрешности всех четырёх формул (просто запустите):

In [ ]:
Ns_plot = 2 ** np.arange(1, 11)
for method, label in [(rect_left, 'левые прямоугольники'), (rect_mid, 'средние прямоугольники'),
                      (trapezoid, 'трапеции'), (simpson, 'Симпсон')]:
    errs = [abs(1 - method(np.sin, 0, np.pi / 2, N)) for N in Ns_plot]
    plt.loglog(Ns_plot, errs, 'o-', ms=4, label=label)
plt.loglog(Ns_plot, 1.0 / Ns_plot, 'k:', lw=1)
plt.loglog(Ns_plot, 1.0 / Ns_plot**2, 'k:', lw=1)
plt.loglog(Ns_plot, 1.0 / Ns_plot**4, 'k:', lw=1)
plt.text(Ns_plot[-1] * 1.1, 1.0 / Ns_plot[-1], '$N^{-1}$')
plt.text(Ns_plot[-1] * 1.1, 1.0 / Ns_plot[-1]**2, '$N^{-2}$')
plt.text(Ns_plot[-1] * 1.1, 1.0 / Ns_plot[-1]**4, '$N^{-4}$')
plt.xlim(1.5, 4000)
plt.xlabel('N')
plt.ylabel('погрешность')
plt.legend()
plt.show()

### Когда порядок не достигается (просто запустите)

Порядок $p$ гарантирован только для достаточно гладких функций. У $\sqrt{x}$ производная в нуле бесконечна:

$$\int_0^1 \sqrt{x}\,dx = \frac{2}{3}.$$

In [ ]:
order_table(simpson, np.sqrt, 0, 1, 2 / 3)

Отношение около $2.83 = 2^{1.5}$: порядок всего $1.5$ вместо $4$.

Бывает и наоборот. Для периодической функции на полном периоде формула трапеций сходится гораздо быстрее, чем $h^2$:

$$\int_0^{2\pi} e^{\cos x}\,dx.$$

In [ ]:
from scipy.special import i0

def f_periodic(x):
    return np.exp(np.cos(x))

I_periodic = 2 * np.pi * i0(1.0)     # точное значение выражается через функцию Бесселя
for N in [2, 4, 8, 16]:
    print(N, abs(I_periodic - trapezoid(f_periodic, 0, 2 * np.pi, N)))

## 5. Формула Рунге–Ромберга

Пока мы находили погрешность, зная точный ответ. Что делать, если он неизвестен?

Пусть формула имеет порядок $p$. Вычислим интеграл на двух сетках: с $N$ отрезками и с $rN$ отрезками (обычно $r = 2$). Тогда

$$\boxed{I = U_{rN} + R_{rN} + O(h^{p+1}), \qquad R_{rN} = \frac{U_{rN} - U_N}{r^p - 1}}$$

- $R_{rN}$ — **оценка погрешности** значения $U_{rN}$, вычисленная без знания точного ответа (апостериорная оценка).
- $U_{rN} + R_{rN}$ — **уточнённое значение**, более высокого порядка точности.

*Откуда формула.* $I = U_N + Ch^p + \dots$ и $I = U_{rN} + C\left(\dfrac{h}{r}\right)^p + \dots$ Вычитая, получаем $C\left(\dfrac{h}{r}\right)^p \approx \dfrac{U_{rN} - U_N}{r^p - 1}$.

In [ ]:
# Пишем вместе:
# 1) формула трапеций при N = 10 и N = 20
# 2) оценка R по формуле Рунге–Ромберга (r = 2, p = 2)
# 3) напечатать оценку R и настоящую погрешность 1 - U_2N


Теперь уточнённое значение.

In [ ]:
# Пишем вместе:
# 1) уточнённое значение U_2N + R_2N и его погрешность
# 2) разность уточнённого значения и формулы Симпсона при N = 20


**Мини-задача.** Для $\displaystyle\int_1^2 \frac{dx}{x}$ вычислите формулу Симпсона при $N = 10$ и $N = 20$, оцените погрешность по формуле Рунге–Ромберга (чему равно $p$?) и сравните с настоящей погрешностью.

In [ ]:
# Ваш код


## 6. Методика Ричардсона

**Задача:** вычислить интеграл с заданной точностью $\varepsilon$, не зная точного ответа.

1. Вычисляем интеграл на последовательности сгущающихся сеток: $N,\ rN,\ r^2N,\ \dots$
2. На каждой новой сетке находим оценку погрешности $R_{rN}$ по формуле Рунге–Ромберга.
3. Останавливаемся, когда $|R_{rN}| < \varepsilon$.

Результат: значение $U$ на последней сетке и оценка его погрешности $R$ (а также уточнённое значение $U + R$).

In [ ]:
# Пишем вместе:
# функция richardson(method, p, f, a, b, eps, N=4, r=2, max_steps=20):
#   U_old — значение на начальной сетке;
#   цикл не более max_steps раз:
#     сгущаем сетку (N = r * N), считаем U, оценку R = (U - U_old) / (r**p - 1);
#     печатаем N, U, R;
#     если |R| < eps, выходим из цикла; иначе U_old = U
#   вернуть U и R


In [ ]:
# Пишем вместе:
# 1) интеграл от sin на [0, pi/2] формулой трапеций (p = 2) с точностью 1e-8
# 2) настоящая погрешность и её оценка


In [ ]:
# Пишем вместе:
# то же по формуле Симпсона (p = 4)


Теперь интеграл, точное значение которого неизвестно:

$$\int_0^1 e^{-x^2}\,dx.$$

In [ ]:
# Пишем вместе:
# 1) функция f_gauss(x) = exp(-x^2)
# 2) интеграл на [0, 1] формулой Симпсона с точностью 1e-10
# 3) уточнённое значение U + R


### Можно ли доверять оценке? Эффективный порядок

Формула Рунге–Ромберга верна, только если погрешность действительно ведёт себя как $Ch^p$. Это можно проверить по трём сеткам, не зная точного ответа:

$$p_{\text{эфф}} = \log_r\frac{U_{rN} - U_N}{U_{r^2N} - U_{rN}}.$$

Если $p_{\text{эфф}}$ близок к теоретическому $p$, оценке можно доверять.

Пример: $\displaystyle\int_0^1\sqrt{x}\,dx$ по формуле Симпсона (просто запустите).

In [ ]:
U_sqrt, R_sqrt = richardson(simpson, 4, np.sqrt, 0, 1, 1e-6)
print('оценка погрешности:   ', R_sqrt)
print('настоящая погрешность:', 2 / 3 - U_sqrt)

U_16, U_32, U_64 = simpson(np.sqrt, 0, 1, 16), simpson(np.sqrt, 0, 1, 32), simpson(np.sqrt, 0, 1, 64)
print('эффективный порядок:  ', np.log2((U_32 - U_16) / (U_64 - U_32)))

Оценка с $p = 4$ занижает погрешность в 8 раз: настоящий порядок равен $1.5$, и эффективный порядок это показывает.

## 7. Интегрирование в `scipy`

- `integrate.quad(f, a, b)` — интеграл от функции. Возвращает значение и оценку погрешности. Внутри используются квадратурные формулы Гаусса с автоматическим выбором узлов.
- `integrate.trapezoid(y, x)` и `integrate.simpson(y, x=x)` — формулы трапеций и Симпсона для табличных данных.

In [ ]:
# Пишем вместе:
# импортировать модуль integrate из scipy и вычислить интеграл от f_gauss на [0, 1] функцией quad


---
## 8. Задачи

### Основные

**Задача 1.** Вычислите $\displaystyle\int_0^1 e^x\,dx = e - 1$ формулами средних прямоугольников, трапеций и Симпсона при $N = 10$. Найдите погрешности (точное значение `np.e - 1`).

In [ ]:
# Ваш код


**Задача 2.** Число $\pi$ можно вычислить как

$$\pi = \int_0^1 \frac{4}{1 + x^2}\,dx.$$

Вычислите его с точностью $10^{-6}$ методикой Ричардсона по формуле трапеций. Сколько отрезков понадобилось? Сравните результат и уточнённое значение с `np.pi`.

In [ ]:
# Ваш код


**Задача 3.** Вычислите $\displaystyle\int_0^1 \sqrt{1 + x^3}\,dx$ с точностью $10^{-8}$ методикой Ричардсона по формуле Симпсона и сравните с результатом `integrate.quad`.

In [ ]:
# Ваш код


**Задача 4. Табличные данные.** Скорость автомобиля измеряли каждые 2 секунды в течение 20 секунд (таблица ниже). Найдите пройденный путь с помощью `integrate.trapezoid` и `integrate.simpson`.

In [ ]:
t_table = np.arange(0, 21, 2.0)                                                         # время, с
v_table = np.array([0.0, 4.1, 7.8, 11.0, 13.6, 15.7, 17.3, 18.5, 19.4, 20.0, 20.4])     # скорость, м/с

In [ ]:
# Ваш код


**Задача 5. Возвращаемся к разминке.** Найдите $x$, при котором

$$\int_0^x e^{-t^2}\,dt = 0.5.$$

Указание: задайте функцию `F(x)`, которая вычисляет интеграл формулой Симпсона (например, при $N = 100$) и вычитает $0.5$, и найдите её корень с помощью `brentq` на отрезке $[0, 1]$.

In [ ]:
# Ваш код


### Для тех, кто справился быстрее

**Задача 6. Период маятника.** Период колебаний математического маятника с амплитудой $\theta_0$ (не обязательно малой) равен

$$\frac{T}{T_0} = \frac{2}{\pi}\int_0^{\pi/2}\frac{d\varphi}{\sqrt{1 - k^2\sin^2\varphi}}, \qquad k = \sin\frac{\theta_0}{2},$$

где $T_0 = 2\pi\sqrt{l/g}$ — период малых колебаний. Вычислите $T/T_0$ для амплитуд $10°$, $45°$, $90°$ и $150°$ (перевод в радианы: `np.radians`). При какой из этих амплитуд формула малых колебаний ещё даёт ошибку меньше $1\%$?

In [ ]:
# Ваш код


**Задача 7. Эффективный порядок.** Для $\displaystyle\int_0^1\sqrt{x}\,dx$ и формулы трапеций:
1. Найдите эффективный порядок по сеткам $N = 16, 32, 64$.
2. Запустите `richardson` с точностью $10^{-6}$ сначала с теоретическим порядком $p = 2$, затем с найденным эффективным порядком. В каком случае оценка $R$ ближе к настоящей погрешности $2/3 - U$?

In [ ]:
# Ваш код


---
## Итоги

| Формула | Порядок $p$ | Погрешность при удвоении $N$ |
|---|---|---|
| Левые прямоугольники | 1 | уменьшается в 2 раза |
| Средние прямоугольники | 2 | в 4 раза |
| Трапеции | 2 | в 4 раза |
| Симпсон | 4 | в 16 раз |

- Формула Рунге–Ромберга по двум сеткам даёт оценку погрешности $R_{rN} = \dfrac{U_{rN} - U_N}{r^p - 1}$ и уточнённое значение $U_{rN} + R_{rN}$.
- Методика Ричардсона: сгущаем сетку, пока оценка погрешности не станет меньше заданной точности.
- Порядок достигается только для гладких функций; эффективный порядок позволяет это проверить.

---
## 9. Домашнее задание

### Основная часть

**ДЗ 1. Период маятника.** Используя формулу из задачи 6, постройте график зависимости $T/T_0$ от амплитуды $\theta_0$ в диапазоне от $1°$ до $170°$. На том же графике нарисуйте приближение $1 + \theta_0^2/16$ ($\theta_0$ в радианах). С помощью `brentq` найдите амплитуду, при которой период отличается от $T_0$ ровно на $1\%$.

**ДЗ 2. Цена точности.** Для $\displaystyle\int_0^1 e^x\,dx$ определите методикой Ричардсона, сколько отрезков нужно формулам средних прямоугольников, трапеций и Симпсона, чтобы достичь точности $10^{-8}$. Сделайте вывод.

**ДЗ 3. Таблица и сплайн.** Функция $\sin x$ задана таблицей в 7 равноотстоящих точках на $[0, \pi]$. Вычислите интеграл тремя способами: `integrate.trapezoid`, `integrate.simpson` и через кубический сплайн (`CubicSpline(x, y).integrate(0, np.pi)`). Сравните с точным значением $2$.

### Задачи повышенной сложности (по желанию)

**ДЗ 4. Метод Ромберга.** Уточнение по Рунге–Ромбергу можно применять повторно. Пусть $T_{k,0}$ — формула трапеций на сетке из $2^k$ отрезков, а

$$T_{k,m} = T_{k,m-1} + \frac{T_{k,m-1} - T_{k-1,m-1}}{4^m - 1}.$$

Постройте таблицу $T_{k,m}$ для $\displaystyle\int_0^{\pi/2}\sin x\,dx$ при $k = 0, \dots, 5$ и напечатайте погрешности. Какой столбец соответствует формуле Симпсона? Какая точность достигается при 33 вычислениях функции?

**ДЗ 5. Формулы Гаусса.** Узлы и веса формулы Гаусса на $[-1, 1]$ даёт функция `np.polynomial.legendre.leggauss(n)`. Отрезок $[a, b]$ отображается на $[-1, 1]$ заменой $x = \dfrac{a + b}{2} + \dfrac{b - a}{2}t$. Вычислите $\displaystyle\int_0^1 e^x\,dx$ формулами Гаусса с $n = 2, 3, 5$ узлами и сравните погрешность с формулой Симпсона при том же числе вычислений функции.

**ДЗ 6. Периодические функции.** Исследуйте сходимость формул трапеций и Симпсона для $\displaystyle\int_0^{2\pi} e^{\cos x}\,dx$ (точное значение `2 * np.pi * scipy.special.i0(1.0)`). Какая из формул точнее? Попробуйте объяснить результат.